# Paired empty-vial transition analysis

This notebook extends the existing transition analysis for the CSI dataset in
`data/csi-vial-fill-dataset/vial_fill/labels`.

It makes two focused checks:

1. Pair each filled-vial color with the **same vial immediately before filling**.
   The paired empty Lab offset is removed before calculating ΔE00. A missing or
   unstable empty measurement stays missing; another vial's background is never
   substituted.
2. Test the observation that a darker target recipe, or a target recipe with
   more milliliters in its dominant tank, reaches its acceptable range in fewer
   vials. The association is reported as exploratory because the supplied labels
   cover one production video rather than independent sessions.

The additive Lab correction addresses vial tint and local lighting changes. It
does not replace camera calibration or prove an optical mixing law.

In [1]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Image, display

ROOT = Path.cwd()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from src.color_monitor import config as C
from src.color_monitor.audit import build_vial_table, load_catalog
from src.color_monitor.changeover import vial_report
from src.color_monitor.extract import extract_frames
from src.color_monitor.overlay import save_paired_measurement_overlays
from src.color_monitor.paired_transition import (
    add_paired_empty_correction,
    attach_labeled_prefill_measurements,
    corrected_vial_report,
    test_darkness_volume_hypothesis,
    transition_effect_table,
)
from src.color_monitor.segment import measure_cycles

DATASET = ROOT / "data" / "csi-vial-fill-dataset"
VIAL_FILL = DATASET / "vial_fill"
LABELS = VIAL_FILL / "labels"
OUT = ROOT / "outputs" / "paired_empty_transition"
OUT.mkdir(parents=True, exist_ok=True)
print("Dataset:", DATASET.relative_to(ROOT))

Dataset: data/csi-vial-fill-dataset


## 1. Validate the CSI dataset

In [2]:
required = [
    LABELS / "video5_vial_recipes.csv",
    LABELS / "video5_frame_labels.csv",
    VIAL_FILL / "videos" / "vial_fill_video_5.mp4",
]
missing = [p for p in required if not p.is_file()]
DATA_AVAILABLE = not missing
if missing:
    print("DATASET NOT PRESENT IN THIS CHECKOUT")
    for item in missing:
        print(" -", item.relative_to(ROOT))
else:
    print("All required video-5 inputs are present.")
    for item in required:
        print(" -", item.relative_to(ROOT), f"({item.stat().st_size:,} bytes)")

DATASET NOT PRESENT IN THIS CHECKOUT
 - data/csi-vial-fill-dataset/vial_fill/labels/video5_vial_recipes.csv
 - data/csi-vial-fill-dataset/vial_fill/labels/video5_frame_labels.csv
 - data/csi-vial-fill-dataset/vial_fill/videos/vial_fill_video_5.mp4


## 2. Measure filled and paired pre-fill colors

The video is sampled at the configured interval. For each labeled vial, the
pre-fill measurement uses only the three seconds immediately before that vial's
labeled `start_s`; this avoids selecting the previously filled vial while the
next vial waits beside it. The settled window supplies the filled color.

In [3]:
frames = cycles = vials = pd.DataFrame()
if DATA_AVAILABLE:
    frame_cache = OUT / "video5_frames.csv"
    if frame_cache.is_file():
        frames = pd.read_csv(frame_cache)
        print("Loaded cached frame measurements:", len(frames))
    else:
        frames = extract_frames(video_ids=(5,))
        frames.to_csv(frame_cache, index=False)
        print("Measured and cached frames:", len(frames))
    cycles = measure_cycles(frames)
    vials = build_vial_table(cycles)
    vials = attach_labeled_prefill_measurements(frames, vials, window_s=3.0)
    display(vials[[
        "vial_id", "run_id", "recipe_id", "position_in_run", "L", "a", "b",
        "empty_L", "empty_a", "empty_b", "empty_n_frames", "empty_spread_de",
        "prefill_source", "status",
    ]])
else:
    print("Skipped: copy the CSI dataset to", DATASET.relative_to(ROOT))

Skipped: copy the CSI dataset to data/csi-vial-fill-dataset


## 3. Apply same-vial background correction

In [4]:
corrected = corrected_report = corrected_refs = pd.DataFrame()
canonical_empty = None
if DATA_AVAILABLE:
    catalog = load_catalog()
    raw_report = vial_report(vials, catalog)
    corrected, canonical_empty = add_paired_empty_correction(vials)
    corrected_report, corrected_refs = corrected_vial_report(corrected, catalog)
    comparison = raw_report[["vial_id", "delta_e00_to_recipe", "verdict"]].merge(
        corrected_report, on="vial_id", how="outer"
    ).merge(
        corrected[[
            "vial_id", "empty_delta_e00_to_canonical",
            "background_adjustment_delta_e00", "empty_pair_reason"
        ]],
        on=["vial_id", "empty_pair_reason"], how="left"
    )
    print(
        "Canonical empty Lab:",
        "unavailable" if canonical_empty is None else np.round(canonical_empty, 3),
    )
    print(
        "Usable paired empty measurements:",
        f"{int(corrected.empty_pair_usable.sum())}/{len(corrected)}",
    )
    print("Pair outcomes:")
    display(corrected.empty_pair_reason.value_counts(dropna=False).rename("vials"))
    print("Recipe-reference availability:")
    display(pd.DataFrame({
        "raw_delta_e_available": [comparison.delta_e00_to_recipe.notna().sum()],
        "corrected_delta_e_available": [
            comparison.corrected_delta_e00_to_recipe.notna().sum()
        ],
        "both_available": [comparison[[
            "delta_e00_to_recipe", "corrected_delta_e00_to_recipe"
        ]].notna().all(axis=1).sum()],
    }))
    display(corrected_refs[["n_vials", "status", "tolerance_de"]])
    display(comparison)
    corrected.to_csv(OUT / "paired_vial_colors.csv", index=False)
    comparison.to_csv(OUT / "raw_vs_corrected_delta_e.csv", index=False)
else:
    comparison = pd.DataFrame()
    print("Skipped because the CSI dataset is absent.")

Skipped because the CSI dataset is absent.


## 4. Check the different-vial explanation

`empty_delta_e00_to_canonical` measures how unusual the vial looked in its
labeled pre-fill window. If a final vial rises in L, a, and b because that vial
itself is different, its pre-fill distance should also be unusual and the
corrected filled-vial ΔE should shrink. The contact sheet says `PREFILL`, because
timing establishes that dispensing has not started but does not prove that the
vial contains zero residual liquid.

In [5]:
if DATA_AVAILABLE:
    frame_manifest_columns = [
        "video_id", "cycle", "vial_id", "recipe_id", "start_s", "t_start",
        "empty_t_start", "empty_t_end",
        "empty_t_measure", "stable_t_start", "stable_t_end", "stable_t_measure",
        "empty_L", "empty_a", "empty_b", "L", "a", "b", "prefill_source",
    ]
    frame_manifest = vials[frame_manifest_columns].copy()
    timed_pairs = frame_manifest.dropna(subset=[
        "empty_t_start", "empty_t_end", "empty_t_measure", "start_s",
        "stable_t_start", "stable_t_end", "stable_t_measure",
    ])
    invalid_empty_windows = timed_pairs[
        (timed_pairs.empty_t_start > timed_pairs.empty_t_measure)
        | (timed_pairs.empty_t_measure > timed_pairs.empty_t_end)
        | (timed_pairs.empty_t_end >= timed_pairs.start_s)
        | (timed_pairs.empty_t_start < timed_pairs.start_s - 3.0)
    ]
    if not invalid_empty_windows.empty:
        raise ValueError(
            "A PREFILL window is outside the three seconds immediately before "
            "the labeled fill start_s."
        )
    frame_manifest.to_csv(OUT / "paired_measurement_frame_manifest.csv", index=False)
    overlay_path = save_paired_measurement_overlays(
        vials, OUT / "paired_empty_filled_roi_overlays.png", pairs_per_row=2
    )
    display(frame_manifest)
    if overlay_path is None:
        print("No empty/filled frame pair could be decoded from the video.")
    else:
        print(
            "Each pair shows the labeled PREFILL frame on the left and "
            "FILLED frame on the right. The yellow rectangle is the measured ROI."
        )
        display(Image(filename=str(overlay_path)))
else:
    print("Skipped frame audit because the CSI dataset is absent.")

Skipped frame audit because the CSI dataset is absent.


In [6]:
if DATA_AVAILABLE:
    measured_pairs = comparison[comparison.empty_pair_usable].copy()
    paired_reference_delta = measured_pairs.dropna(subset=[
        "delta_e00_to_recipe", "corrected_delta_e00_to_recipe"
    ]).copy()
    paired_reference_delta["delta_e_reduction"] = (
        paired_reference_delta.delta_e00_to_recipe
        - paired_reference_delta.corrected_delta_e00_to_recipe
    )
    print(f"Measured same-vial pairs: {len(measured_pairs)}")
    print(f"Pairs with both recipe-reference ΔE values: {len(paired_reference_delta)}")
    display(measured_pairs.sort_values(
        "empty_delta_e00_to_canonical", ascending=False
    ))

    if measured_pairs.empty:
        print("No same-vial empty/filled pairs were measured; see pair outcomes above.")
    else:
        fig, axes = plt.subplots(2, 1, figsize=(11, 8), sharex=True)
        if paired_reference_delta.empty:
            axes[0].text(
                .5, .5,
                "Paired colors were measured, but no recipe reference ΔE is available. "
                "See the reference-status table above.",
                ha="center", va="center", transform=axes[0].transAxes,
            )
        else:
            axes[0].plot(
                paired_reference_delta.vial_id,
                paired_reference_delta.delta_e00_to_recipe,
                "o-", label="raw ΔE00",
            )
            axes[0].plot(
                paired_reference_delta.vial_id,
                paired_reference_delta.corrected_delta_e00_to_recipe,
                "o-", label="paired-empty corrected ΔE00",
            )
            axes[0].legend()
        axes[0].set_ylabel("ΔE00 to recipe reference")
        axes[0].grid(alpha=.25)
        axes[1].bar(
            measured_pairs.vial_id,
            measured_pairs.empty_delta_e00_to_canonical,
            label="empty-vial difference",
        )
        axes[1].plot(
            measured_pairs.vial_id,
            measured_pairs.background_adjustment_delta_e00,
            "o-", color="tab:red", label="filled-color correction size",
        )
        axes[1].set_ylabel("ΔE00")
        axes[1].set_xlabel("Vial ID")
        axes[1].legend()
        axes[1].grid(axis="y", alpha=.25)
        fig.suptitle("Raw versus same-vial-background-corrected color")
        fig.tight_layout()
        fig.savefig(OUT / "raw_vs_corrected_delta_e.png", dpi=160)
        plt.show()
else:
    print("Skipped because the CSI dataset is absent.")

Skipped because the CSI dataset is absent.


## 5. Test darkness and dominant-volume recovery

For every recipe change, recovery is the number of vials before the first
corrected in-range vial. `target_darkness = 100 - reference L`. The dominant
volume is the largest of T100/T200/T400 in the target recipe. A negative
Spearman correlation supports the proposed direction: higher darkness or
dominant volume corresponds to fewer pre-acceptable vials.

In [7]:
transitions = hypothesis = pd.DataFrame()
if DATA_AVAILABLE:
    transitions = transition_effect_table(
        corrected_report, corrected_refs, catalog, consecutive=1
    )
    hypothesis = test_darkness_volume_hypothesis(transitions)
    display(transitions)
    display(hypothesis)
    transitions.to_csv(OUT / "transition_effects.csv", index=False)
    hypothesis.to_csv(OUT / "darkness_volume_hypothesis.csv", index=False)
else:
    print("Skipped because the CSI dataset is absent.")

Skipped because the CSI dataset is absent.


In [8]:
if DATA_AVAILABLE and not transitions.empty:
    fig, axes = plt.subplots(1, 2, figsize=(11, 4.5), sharey=True)
    observed = transitions.dropna(subset=["pre_acceptable_vials"])
    axes[0].scatter(observed.target_darkness, observed.pre_acceptable_vials)
    axes[0].set_xlabel("Target darkness (100 - corrected reference L)")
    axes[1].scatter(observed.dominant_ml, observed.pre_acceptable_vials)
    axes[1].set_xlabel("Dominant tank volume (mL)")
    axes[0].set_ylabel("Vials before acceptable range")
    for axis in axes:
        axis.grid(alpha=.25)
    fig.suptitle("Exploratory target strength versus changeover recovery")
    fig.tight_layout()
    fig.savefig(OUT / "target_strength_vs_recovery.png", dpi=160)
    plt.show()
else:
    print("No measured transitions available for plotting.")

No measured transitions available for plotting.


## Interpretation rules

- A final vial with unusual empty-vial ΔE and a large reduction after correction
  supports the different-vial explanation shown in the supplied plot.
- A negative Spearman rho supports the proposed darkness/volume direction in
  this recording. It does not establish causation: target color, composition,
  dispense order, prior recipe, and transition direction can be confounded.
- Results from one labeled video are exploratory. Production use still requires
  repeated directions across independent sessions, stable camera settings, and
  independent clean references.
- Rows without the same vial's stable pre-fill measurement remain unavailable;
  the notebook does not invent or borrow a background.